# Amazon | Desempenho do portfólio e percepção dos clientes

**Pergunta de negócio:** quais produtos, categorias e políticas de desconto merecem atenção a partir do retrato disponível do catálogo?

Execute as células em ordem. Use um kernel Python 3.10+ com `pandas>=2,<4` e `numpy>=1.24`. Se precisar, execute `%pip install 'pandas>=2,<4' 'numpy>=1.24'` em uma célula. Abra este arquivo dentro da árvore do projeto. O CSV original é lido em modo somente leitura.

O enunciado traz “FASE 03” na capa, apesar do nome do PDF indicar Fase 4. Este projeto segue o conteúdo das páginas 2 a 7. Consulte o guia do Power BI para a matriz dos nove critérios e o roteiro executivo.

In [1]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import csv
import hashlib
import json
import re
import sys

import numpy as np
import pandas as pd

inicio = Path.cwd().resolve()
projeto = next((p for p in (inicio, *inicio.parents) if (p / "data/amazon.csv").is_file()), None)
if projeto is None:
    raise FileNotFoundError("Abra o Jupyter a partir da pasta do projeto ou da pasta notebook.")
sys.path.insert(0, str(projeto / "notebook"))
from analise_funcoes import prepare

origem = projeto / "data" / "amazon.csv"
saida = projeto / "powerbi" / "dados_powerbi"
documentos = projeto / "documentos"
conteudo = origem.read_bytes()
assinatura = hashlib.sha256(conteudo).hexdigest()
codificacao = "utf-8-sig"
amostra = conteudo.decode(codificacao)
delimitador = csv.Sniffer().sniff(amostra.splitlines()[0], delimiters=",;\t|").delimiter
bruto = pd.read_csv(origem, sep=delimitador, encoding=codificacao, dtype="string", keep_default_na=False)
faltantes = set(prepare.COLUNAS_OBRIGATORIAS) - set(bruto.columns)
assert not faltantes, f"Colunas obrigatórias ausentes: {faltantes}"
print(f"{len(bruto):,} linhas × {bruto.shape[1]} colunas | UTF-8 | delimitador {delimitador!r}")
display(bruto[["product_id", "category", "actual_price", "discounted_price", "rating", "rating_count"]].head())

1,465 linhas × 16 colunas | UTF-8 | delimitador ','


,product_id,category,actual_price,discounted_price,rating,rating_count
0,B07JW9H4J1,Computers&Accessories|Accessories&Peripherals|...,"₹1,099",₹399,4.2,"24,269"
1,B098NS6PVG,Computers&Accessories|Accessories&Peripherals|...,₹349,₹199,4.0,"43,994"
2,B096MSW6CT,Computers&Accessories|Accessories&Peripherals|...,"₹1,899",₹199,3.9,"7,928"
3,B08HDJ86NZ,Computers&Accessories|Accessories&Peripherals|...,₹699,₹329,4.2,"94,363"
4,B08CF3B7N1,Computers&Accessories|Accessories&Peripherals|...,₹399,₹154,4.2,"16,905"


## 1. Diagnóstico

Cada linha contém um registro de catálogo de um produto, com campos de avaliações concatenadas. O mesmo identificador pode aparecer em mais de um registro. As vírgulas internas aos comentários impedem separar com segurança os textos em avaliações individuais. A nota e a contagem se referem ao produto.

No CSV, a ausência chega como campo vazio, espaço ou marcador. A tabela reconcilia essas ocorrências com os nulos que seriam inferidos pelo leitor padrão. O caractere `|` isolado é tratado como nota inválida. Dentro da categoria, ele delimita a hierarquia.

In [2]:
marcadores = {"na", "n/a", "null", "none", "nan", "<na>", "n/d", "-"}
sem_espacos = bruto.apply(lambda col: col.str.strip())
vazios = sem_espacos.eq("")
ausentes_texto = sem_espacos.apply(lambda col: col.str.casefold().isin(marcadores))
dados = sem_espacos.mask(vazios | ausentes_texto)
nulos_leitura_padrao = pd.read_csv(origem, sep=delimitador, encoding=codificacao, dtype="string").isna().sum()
perfil = pd.DataFrame({
    "tipo_original": bruto.dtypes.astype(str),
    "nulos_leitura_padrao": nulos_leitura_padrao,
    "campos_vazios": vazios.sum(),
    "marcadores_ausencia": ausentes_texto.sum(),
    "total_ausente": dados.isna().sum(),
    "percentual_ausente": dados.isna().mean().mul(100).round(3),
})
display(perfil)
print(f"Linhas idênticas: {bruto.duplicated().sum()}.")
print(f"Registros excedentes por product_id: {dados['product_id'].dropna().duplicated().sum()}.")
print("Símbolos observados nos preços:", sorted(set(re.findall(r"[^\d.,\s]", "".join(dados["actual_price"].dropna())))))

,tipo_original,nulos_leitura_padrao,campos_vazios,marcadores_ausencia,total_ausente,percentual_ausente
product_id,string,0,0,0,0,0.000
product_name,string,0,0,0,0,0.000
category,string,0,0,0,0,0.000
discounted_price,string,0,0,0,0,0.000
actual_price,string,0,0,0,0,0.000
discount_percentage,string,0,0,0,0,0.000
rating,string,0,0,0,0,0.000
rating_count,string,2,2,0,2,0.137
about_product,string,0,0,0,0,0.000
user_id,string,0,0,0,0,0.000


Linhas idênticas: 0.
Registros excedentes por product_id: 114.
Símbolos observados nos preços: ['₹']


## 2. Tipos, integridade e granularidade

Use uma linha coerente por SKU. Quando há vários registros, priorize um conjunto comercial válido, depois a presença da nota, a maior contagem válida e a ordem original como desempate. A contagem máxima prioriza a observação com mais evidência e o desempate dá reprodutibilidade. Sem uma data, a política pode privilegiar as contagens maiores e constitui uma escolha analítica. O notebook compara a escolha com a primeira linha de cada SKU.

Converta preços em rupias indianas (INR), descontos em frações e contagens em inteiros. Preserve ausências. Mantenha o desconto informado como métrica principal e audite diferenças acima de dois pontos percentuais em relação ao desconto implícito nos preços.

In [3]:
validos = dados.dropna(subset=["product_id"]).drop_duplicates().copy()
numeros = pd.DataFrame(index=validos.index)
for campo in ["actual_price", "discounted_price", "discount_percentage", "rating", "rating_count"]:
    texto_campo = validos[campo].str.replace(r"[₹%,\s]", "", regex=True)
    numeros[campo] = pd.to_numeric(texto_campo, errors="coerce").astype("float64")

definicoes = {
    "actual_price": numeros["actual_price"].gt(0),
    "discounted_price": numeros["discounted_price"].ge(0),
    "discount_percentage": numeros["discount_percentage"].between(0, 100),
    "rating": numeros["rating"].between(1, 5),
    "rating_count": numeros["rating_count"].ge(0) & numeros["rating_count"].mod(1).eq(0),
}
intervalos = pd.DataFrame(definicoes) & np.isfinite(numeros)
invalidos = validos[numeros.columns].notna() & ~intervalos
diagnostico = pd.DataFrame({"invalidos": invalidos.sum(), "ausentes": validos[numeros.columns].isna().sum()})
display(diagnostico)
numeros = numeros.where(intervalos)
comerciais = ["actual_price", "discounted_price", "discount_percentage"]
chaves = validos.assign(
    comercial_valido=numeros[comerciais].notna().all(axis=1),
    nota_valida=numeros["rating"].notna(),
    contagem=numeros["rating_count"],
)
ordem = chaves.sort_values(
    ["comercial_valido", "nota_valida", "contagem"], ascending=False, kind="stable"
)
indices = ordem.drop_duplicates("product_id").index
primeiros = validos.drop_duplicates("product_id").index
selecionados = validos.loc[indices].sort_index().copy()
selecionados[numeros.columns] = numeros.loc[selecionados.index]
fato, _, auditoria = prepare.limpar_produtos(selecionados)
fato["rating_count"] = fato["rating_count"].astype("Int64")
fato["linhas_origem"] = fato["product_id"].map(dados["product_id"].value_counts()).astype("int64")
fato["rotulo_produto"] = fato["product_name"].fillna("Produto") + " [" + fato["product_id"] + "]"
fato["ordem_desconto"] = fato["faixa_desconto"].map(
    {faixa: i for i, faixa in enumerate(prepare.cfg.ROTULOS_DESCONTO, 1)}
).fillna(99).astype("int64")
fato["tem_descricao"] = fato["about_product"].notna().astype("int64")
fato["about_product"] = fato["about_product"].str.slice(0, 500)

conflitos = validos.groupby("product_id")[["category", *numeros.columns]].nunique()
display(pd.DataFrame({
    "etapa": ["Origem", "Com identificador", "Sem duplicatas integrais", "Um registro por SKU"],
    "linhas": [len(dados), int(dados["product_id"].notna().sum()), len(validos), len(fato)],
}))
display(conflitos.gt(1).sum().rename("SKUs com valores divergentes").to_frame())
base_sensibilidade = pd.DataFrame({
    "regra": ["Primeira linha válida por ID", "Seleção analítica"],
    "avaliacoes": [numeros.loc[primeiros, "rating_count"].sum(), fato["rating_count"].sum()],
    "nota_media": [numeros.loc[primeiros, "rating"].mean(), fato["rating"].mean()],
})
display(base_sensibilidade)
print("Diferenças de desconto maiores que 2 pp nos SKUs selecionados:",
      auditoria["desconto_inconsistente_mais_de_2pp"])

,invalidos,ausentes
actual_price,0,0
discounted_price,0,0
discount_percentage,0,0
rating,1,0
rating_count,0,2


,etapa,linhas
0,Origem,1465
1,Com identificador,1465
2,Sem duplicatas integrais,1465
3,Um registro por SKU,1351


,SKUs com valores divergentes
category,0
actual_price,1
discounted_price,3
discount_percentage,4
rating,0
rating_count,31


,regra,avaliacoes,nota_media
0,Primeira linha válida por ID,23802423.0,4.091852
1,Seleção analítica,23802463.0,4.091852


Diferenças de desconto maiores que 2 pp nos SKUs selecionados: 0


## 3. Customer Insights e segmentos

Remova blocos de comentários idênticos no mesmo SKU antes de agregar as contribuições. O dicionário em `techchallenge/config.py` identifica temas de forma exploratória. Conte menções por mil palavras para comparar grupos com volumes diferentes de texto. Um bloco pode conter elogios e queixas. Valide a interpretação com leitura humana de exemplos e trate a nota como classificação do produto.

Os cortes são calculados no portfólio inteiro e permanecem fixos quando o relatório recebe filtros. O rating ajustado aproxima a nota de produtos com pouca evidência da média global.

In [4]:
def juntar_blocos(serie):
    return " ".join(serie.dropna().drop_duplicates())

texto = validos.groupby("product_id", sort=False, as_index=False).agg(
    review_title=("review_title", juntar_blocos),
    review_content=("review_content", juntar_blocos),
)
metricas = prepare.metricas_de_texto(texto)
fato = fato.merge(metricas, on="product_id", validate="one_to_one")
fato = fato.merge(texto[["product_id", "review_title"]], on="product_id", validate="one_to_one")
fato["review_title"] = fato["review_title"].str.slice(0, 300)
fato["tem_comentario"] = fato["n_palavras"].gt(0).astype("int64")
fato["rating_count"] = fato["rating_count"].astype(float)
fato, parametros = prepare.classificar(fato)
fato["rating_count"] = fato["rating_count"].astype("Int64")
fato_tema, dim_tema = prepare.tabela_temas(texto)
colunas_fato = [c for c in fato.columns if c not in {"img_link", "product_link"}]
fato = fato[colunas_fato]
display(parametros)
display(fato["quadrante"].value_counts().to_frame("produtos"))
assert fato["product_id"].is_unique and fato["product_id"].notna().all()
assert fato["linhas_origem"].sum() == dados["product_id"].notna().sum()
assert fato["rating"].dropna().between(1, 5).all()
assert fato["discount_percentage"].dropna().between(0, 1).all()
assert fato["rating_count"].dropna().ge(0).all()
assert fato["actual_price"].dropna().gt(0).all()
assert fato["discounted_price"].dropna().ge(0).all()
assert (fato["discounted_price"] <= fato["actual_price"]).where(
    fato[["actual_price", "discounted_price"]].notna().all(axis=1), True
).all(), "Há preços finais acima dos preços originais."
assert not fato_tema.duplicated(["product_id", "tema"]).any()
assert fato_tema["product_id"].isin(fato["product_id"]).all()
assert fato_tema["tema"].isin(dim_tema["tema"]).all()
assert dim_tema["tema"].is_unique and fato_tema["mencoes"].ge(0).all()
assert len(fato_tema) == len(fato) * len(dim_tema)
display(fato.isna().sum().loc[lambda s: s.gt(0)].to_frame("ausentes após tratamento"))

,parametro,valor,descricao
0,rating_medio_geral,4.091852,Média simples de rating (C do rating ajustado)
1,rating_count_mediano,4740.000000,Mediana de rating_count (m do rating ajustado)
2,rating_q_baixo,3.900000,Quantil 25% de rating: abaixo = grupo_nota Baixa
3,rating_q_alto,4.300000,Quantil 75% de rating: a partir daqui = grupo_...
4,engajamento_q_baixo,1106.000000,Quantil 25% de rating_count: até aqui = Baixo
5,engajamento_q_alto,16020.000000,Quantil 75% de rating_count: a partir daqui = ...


,produtos
quadrante,
Demais,1134
Estrela,112
Joia escondida,81
Alerta,21
Sem dados,3


,ausentes após tratamento
rating,1
rating_count,2
saldo_sentimento,2
rating_ajustado,3


## 4. Análise exploratória

Compare volume e satisfação por categoria. Use a nota ponderada para representar o peso das avaliações agregadas e mostre também a média simples por SKU. Os grupos com poucos produtos pedem leitura junto com o tamanho da amostra.

Engajamento é a contagem acumulada de avaliações. A base é transversal e carece de vendas, impressões, margens, datas e avaliação individual vinculada ao comentário. Use correlações como descrição da amostra, sem atribuir o resultado ao desconto.

In [5]:
def nota_ponderada(grupo):
    pares = grupo.dropna(subset=["rating", "rating_count"])
    pesos = pares["rating_count"].astype(float)
    return np.average(pares["rating"], weights=pesos) if pesos.sum() else np.nan

categorias = fato.groupby("categoria_nivel1").agg(
    produtos=("product_id", "size"),
    avaliacoes=("rating_count", "sum"),
    nota_media=("rating", "mean"),
    desconto_medio=("discount_percentage", "mean"),
)
categorias["nota_ponderada"] = fato.groupby("categoria_nivel1")[["rating", "rating_count"]].apply(nota_ponderada)
categorias = categorias.sort_values("avaliacoes", ascending=False)
display(categorias.round(4))

def barras(serie, titulo):
    teto = max(float(serie.max()), 1)
    linhas = []
    for nome, valor in serie.items():
        largura = max(0, float(valor) / teto * 100)
        linhas.append(
            f"<div style='display:grid;grid-template-columns:210px 1fr 85px;gap:12px;margin:7px 0'>"
            f"<span>{escape(str(nome))}</span><span style='background:#edf2f5'>"
            f"<span style='display:block;width:{largura:.2f}%;height:16px;background:#147d78'></span>"
            f"</span><span>{valor:,.1f}</span></div>"
        )
    display(HTML(f"<div style='max-width:790px'><strong>{escape(titulo)}</strong>{''.join(linhas)}</div>"))

barras(categorias["avaliacoes"].head(6), "Contagem acumulada por categoria")
total = int(fato["rating_count"].sum())
top10 = float(fato["rating_count"].nlargest(10).sum() / total)
print(f"Os 10 SKUs de maior engajamento concentram {top10:.1%} das avaliações.")
print(f"A categoria líder é {categorias.index[0]}, com {categorias.iloc[0]['avaliacoes'] / total:.1%} das avaliações.")
campos_prioridade = ["product_id", "product_name", "categoria_nivel1", "rating", "rating_count", "rating_ajustado"]
for quadrante in ["Alerta", "Joia escondida", "Estrela"]:
    grupo = fato.loc[fato["quadrante"].eq(quadrante)]
    campo_ordem = "rating_ajustado" if quadrante == "Joia escondida" else "rating_count"
    print(f"{quadrante}: {len(grupo)} produtos.")
    display(grupo.nlargest(5, campo_ordem)[campos_prioridade])

,produtos,avaliacoes,nota_media,desconto_medio,nota_ponderada
categoria_nivel1,,,,,
Electronics,490,14208441,4.0778,0.4991,4.1428
Computers & Accessories,375,6335182,4.1539,0.5322,4.2317
Home & Kitchen,448,2991069,4.0407,0.4012,4.0923
Office Products,31,149675,4.3097,0.1235,4.3272
Musical Instruments,2,88882,3.9000,0.4600,3.9545
Toys & Games,1,15867,4.3000,0.0000,4.3000
Home Improvement,2,8566,4.2500,0.5750,4.3493
Health & Personal Care,1,3663,4.0000,0.5300,4.0000
Car & Motorbike,1,1118,3.8000,0.4200,3.8000


[Gráfico]

Os 10 SKUs de maior engajamento concentram 15.2% das avaliações.
A categoria líder é Electronics, com 59.7% das avaliações.
Alerta: 21 produtos.


,product_id,product_name,categoria_nivel1,rating,rating_count,rating_ajustado
554,B08JQN8DGZ,boAt Airdopes 121v2 in-Ear True Wireless Earbu...,Electronics,3.8,180998,3.807448
605,B07LG59NPV,Boult Audio Probass Curve Bluetooth Wireless i...,Electronics,3.8,103052,3.812834
651,B08FB2LNSZ,"JBL Tune 215BT, 16 Hrs Playtime with Quick Cha...",Electronics,3.7,87798,3.720072
398,B085W8CFLH,PTron Tangent Lite Bluetooth 5.0 Earphones wit...,Electronics,3.5,83996,3.531615
623,B07L8KNP5F,ZEBRONICS Zeb-Thunder Bluetooth Wireless Over ...,Electronics,3.8,60026,3.821360


Joia escondida: 81 produtos.


,product_id,product_name,categoria_nivel1,rating,rating_count,rating_ajustado
1285,B088WCFPQF,Cafe JEI French Press Coffee and Tea Maker 600...,Home & Kitchen,4.5,1065,4.166732
286,B07RY2X9MP,AmazonBasics 10.2 Gbps High-Speed 4K HDMI Cabl...,Electronics,4.5,1029,4.164652
15,B09C6HXFC1,Duracell USB Lightning Apple Certified (Mfi) B...,Computers & Accessories,4.5,815,4.151733
738,B09MZ6WZ6V,INOVERA World Map Extended Anti Slip Rubber Ga...,Computers & Accessories,4.4,1030,4.146859
488,B0B8CHJLWJ,Kyosei Advanced Tempered Glass Compatible with...,Electronics,4.5,596,4.137440


Estrela: 112 produtos.


,product_id,product_name,categoria_nivel1,rating,rating_count,rating_ajustado
8,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,Electronics,4.4,426973,4.396617
42,B014I8SSD0,"Amazon Basics High-Speed HDMI Cable, 6 Feet - ...",Electronics,4.4,426973,4.396617
60,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",Electronics,4.4,426973,4.396617
551,B005FYNT3G,SanDisk Cruzer Blade 32GB USB Flash Drive,Computers & Accessories,4.3,253105,4.296174
779,B09X7DY7Q4,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,Electronics,4.5,205052,4.490778


In [6]:
numericos = fato[["actual_price", "discount_percentage", "rating"]].assign(
    log_avaliacoes=np.log1p(fato["rating_count"].astype(float))
)
pares = [
    ("discount_percentage", "rating"),
    ("discount_percentage", "log_avaliacoes"),
    ("actual_price", "discount_percentage"),
    ("log_avaliacoes", "rating"),
]
correlacoes = []
for x, y in pares:
    par = numericos[[x, y]].dropna()
    correlacoes.append((x, y, len(par), par[x].corr(par[y]), par[x].rank().corr(par[y].rank())))
correlacoes = pd.DataFrame(correlacoes, columns=["variavel_x", "variavel_y", "n", "pearson", "spearman"])
display(correlacoes.round(4))
faixas = fato.groupby(["ordem_desconto", "faixa_desconto"]).agg(
    produtos=("product_id", "size"),
    preco_original_medio=("actual_price", "mean"),
    preco_final_medio=("discounted_price", "mean"),
    nota_media=("rating", "mean"),
    avaliacoes_medianas=("rating_count", "median"),
).droplevel(0)
display(faixas.round(2))
barras(faixas["produtos"], "Quantidade de produtos por faixa de desconto")
rho = correlacoes.iloc[0]["spearman"]
print(f"Desconto × nota tem Spearman {rho:.3f} na amostra. "
      f"A magnitude é {'pequena' if abs(rho) < 0.3 else 'ao menos moderada'} pela regra descritiva de 0,30.")

,variavel_x,variavel_y,n,pearson,spearman
0,discount_percentage,rating,1350,-0.1619,-0.1497
1,discount_percentage,log_avaliacoes,1349,-0.1330,-0.1311
2,actual_price,discount_percentage,1351,-0.1118,-0.0648
3,log_avaliacoes,rating,1348,0.2466,0.1933


,produtos,preco_original_medio,preco_final_medio,nota_media,avaliacoes_medianas
faixa_desconto,,,,,
1. 0-19%,162,4483.13,3978.69,4.16,6744.5
2. 20-39%,316,9469.11,6629.17,4.13,7235.0
3. 40-59%,449,5684.26,2976.69,4.08,3201.0
4. 60-79%,343,3132.99,980.83,4.05,4269.5
5. 80% ou mais,81,4239.81,645.25,4.02,2451.0


[Gráfico]

Desconto × nota tem Spearman -0.150 na amostra. A magnitude é pequena pela regra descritiva de 0,30.


In [7]:
tema_grupos = fato_tema.merge(
    fato[["product_id", "grupo_nota"]], on="product_id", validate="many_to_one"
)
palavras_por_nota = fato.groupby("grupo_nota")["n_palavras"].sum()
taxas = (
    tema_grupos.groupby(["grupo_nota", "tema"])["mencoes"].sum()
    .div(palavras_por_nota.where(palavras_por_nota.gt(0)), level="grupo_nota").mul(1000)
    .unstack("grupo_nota")[["Alta", "Baixa"]]
)
taxas["dif_baixa_menos_alta"] = taxas["Baixa"] - taxas["Alta"]
display(taxas.sort_values("dif_baixa_menos_alta", ascending=False).round(3))
barras(taxas["Baixa"].sort_values(ascending=False).head(6), "Menções por mil palavras nos produtos de nota baixa")
tema_prioritario = taxas["dif_baixa_menos_alta"].idxmax()
padrao = "|".join(map(re.escape, prepare.cfg.TEMAS[tema_prioritario]))
print(f"Maior diferença pró nota baixa: {tema_prioritario}. Exemplos para leitura de contexto:")
for grupo_nota in ["Baixa", "Alta"]:
    grupo = fato.loc[fato["grupo_nota"].eq(grupo_nota), ["product_id", "rating_count"]]
    blocos = grupo.merge(texto, on="product_id").sort_values("rating_count", ascending=False)
    for linha in blocos.itertuples(index=False):
        match = re.search(padrao, linha.review_content, re.IGNORECASE)
        if match:
            trecho = linha.review_content[max(0, match.start() - 90):match.end() + 150]
            print(grupo_nota, linha.product_id, "→", " ".join(trecho.split())[:250])
            break

grupo_nota,Alta,Baixa,dif_baixa_menos_alta
tema,,,
Preço e custo-benefício,10.473,12.858,2.385
Qualidade e acabamento,8.726,10.435,1.709
Bateria e carregamento,6.755,7.797,1.042
Durabilidade e defeitos,2.037,2.735,0.698
"Entrega, embalagem e atendimento",3.681,3.694,0.013
"Som, imagem e desempenho",9.576,8.972,-0.604
Compatibilidade e conexão,8.343,6.909,-1.434
Instalação e facilidade de uso,6.297,3.958,-2.339


[Gráfico]

Maior diferença pró nota baixa: Preço e custo-benefício. Exemplos para leitura de contexto:
Baixa B08JQN8DGZ → arge is a cherry on top.Overall, I'd say it's performance is very average. Definitely not worth ₹1300 in my opinion.,,Not adaptive for every person's ear. My wife doesn't suit the ear airdopes. Partially satisfied. 799/- rupees i bought. Don't
Alta B014I8SSD0 → of now. Picture quality is also not bad. Don't expect something out of the world at this price point. But you can go for it if you have a tight budget.,Initial impressions: works as said, glad I didn’t go for cheaper options (not that it’s too


## 5. Exportação e reconciliação

Os CSVs são gravados em UTF-8 com BOM e separador vírgula. O Power Query aplica tipos com cultura `en-US` para ler os decimais com ponto. O texto integral das avaliações e os identificadores dos clientes ficam fora do modelo. A coluna de título inclui trechos agregados para leitura qualitativa.

A reconciliação relê os CSVs e confere chaves, quantidades, totais e médias. O arquivo de referência inclui resultados globais e por categoria para testar a propagação dos filtros no Power BI.

In [8]:
saida.mkdir(parents=True, exist_ok=True)
tabelas = {
    "Fato_Produto": fato,
    "Fato_ProdutoTema": fato_tema,
    "Dim_Tema": dim_tema,
    "Parametros": parametros,
}
for nome, tabela in tabelas.items():
    tabela.to_csv(saida / f"{nome}.csv", index=False, encoding="utf-8-sig")

lido = pd.read_csv(saida / "Fato_Produto.csv", dtype={"product_id": "string"})
assert lido["product_id"].tolist() == fato["product_id"].tolist()
assert lido["rating_count"].sum() == total
assert np.isclose(nota_ponderada(lido), nota_ponderada(fato))
assert np.isclose(lido["discount_percentage"].mean(), fato["discount_percentage"].mean())
for nome in ("Fato_ProdutoTema", "Dim_Tema", "Parametros"):
    recarregado = pd.read_csv(saida / f"{nome}.csv")
    assert len(recarregado) == len(tabelas[nome])
    if nome == "Fato_ProdutoTema":
        assert not recarregado.duplicated(["product_id", "tema"]).any()
        assert recarregado["mencoes"].sum() == fato_tema["mencoes"].sum()

def indicadores(grupo):
    return {
        "produtos": int(len(grupo)),
        "categorias": int(grupo["categoria_nivel1"].nunique()),
        "avaliacoes": int(grupo["rating_count"].sum()),
        "nota_media": float(grupo["rating"].mean()),
        "nota_ponderada": float(nota_ponderada(grupo)),
        "desconto_medio": float(grupo["discount_percentage"].mean()),
        "alerta": int(grupo["quadrante"].eq("Alerta").sum()),
        "estrelas": int(grupo["quadrante"].eq("Estrela").sum()),
        "joias": int(grupo["quadrante"].eq("Joia escondida").sum()),
    }

referencia = pd.DataFrame([
    {"filtro": "Portfólio completo", **indicadores(fato)},
    *({"filtro": nome, **indicadores(grupo)} for nome, grupo in fato.groupby("categoria_nivel1")),
])
referencia.to_csv(documentos / "validacao_indicadores.csv", index=False, encoding="utf-8-sig")
perfil.to_csv(documentos / "diagnostico_ausencias.csv", encoding="utf-8-sig", index_label="campo")
auditoria.update({
    "sha256": assinatura,
    "linhas_csv": len(bruto),
    "invalidos_por_campo": {campo: int(qtd) for campo, qtd in invalidos.sum().items()},
    "conflitos_por_campo": {campo: int(qtd) for campo, qtd in conflitos.gt(1).sum().items()},
})
auditoria["linhas_brutas"] = len(bruto)
auditoria["linhas_sem_product_id"] = int(dados["product_id"].isna().sum())
auditoria["linhas_duplicadas_de_product_id"] = int(dados["product_id"].dropna().duplicated().sum())
auditoria["conflitos_numericos_entre_duplicatas"] = int(conflitos[numeros.columns].gt(1).sum().sum())
auditoria["produtos_finais"] = len(fato)
auditoria["rating_invalidos_texto"] = int(invalidos["rating"].sum())
(documentos / "auditoria.json").write_text(json.dumps(auditoria, indent=2, ensure_ascii=False), encoding="utf-8")
assert hashlib.sha256(origem.read_bytes()).hexdigest() == assinatura
display(referencia.round(4))
print("Exportação validada:", saida)

,filtro,produtos,categorias,avaliacoes,nota_media,nota_ponderada,desconto_medio,alerta,estrelas,joias
0,Portfólio completo,1351,9,23802463,4.0919,4.1607,0.4669,21,112,81
1,Car & Motorbike,1,1,1118,3.8000,3.8000,0.4200,0,0,0
2,Computers & Accessories,375,1,6335182,4.1539,4.2317,0.5322,2,56,21
3,Electronics,490,1,14208441,4.0778,4.1428,0.4991,16,51,26
4,Health & Personal Care,1,1,3663,4.0000,4.0000,0.5300,0,0,0
5,Home & Kitchen,448,1,2991069,4.0407,4.0923,0.4012,2,5,33
6,Home Improvement,2,1,8566,4.2500,4.3493,0.5750,0,0,0
7,Musical Instruments,2,1,88882,3.9000,3.9545,0.4600,1,0,0
8,Office Products,31,1,149675,4.3097,4.3272,0.1235,0,0,1
9,Toys & Games,1,1,15867,4.3000,4.3000,0.0000,0,0,0


Exportação validada: powerbi/dados_powerbi


## 6. Decisões para testar

1. Investigue os produtos em **Alerta** começando pelos que acumulam mais avaliações. Cruze o conteúdo dos comentários com a categoria e valide as causas em uma leitura amostral.
2. Teste exposição adicional para **Joias escondidas** com evidência suficiente. Antes do teste, capture impressões, conversão e margem para medir seu efeito.
3. Teste alterações de desconto dentro da mesma categoria e faixa de preço. Um experimento com grupo de comparação permite estimar o efeito sobre conversão e margem.
4. Use os temas mais desbalanceados entre notas altas e baixas como pauta de revisão de cadastro e controle de qualidade. Confira o contexto do termo e corrija descrições quando houver evidência convergente.